# Les 16 - Het implementeren van schaalbare agents met Microsoft Foundry

In dit notitieboek bouw je een **productieklaar klantenservice-agent** voor het fictieve bedrijf **Contoso**. In tegenstelling tot eerdere lessen is het hier niet de redeneerlus van de agent — het is alles wat *rondom* de agent zit en ervoor zorgt dat een agent veilig op schaal kan draaien:

1. **Tool-aanroepen** — bestellingsopzoeken en ticketcreatie.
2. **RAG** — beleidsantwoorden vanuit een kennisbank.
3. **Geheugen** — de klant onthouden over meerdere rondes.
4. **Modelroutering** — eenvoudige aanvragen naar een klein model sturen, complexe naar een groot model.
5. **Responscaching** — herhaalde vragen bedienen zonder modeloproep.
6. **Menselijke goedkeuring** — terugbetalingen boven een drempel pauzeren voor goedkeuring.
7. **Evaluatiepoort** — een offline testset die een slechte release tegenhoudt.
8. **Observeerbaarheid** — OpenTelemetry-tracing rond elke aanvraag.

Elke sectie is zelfstandig en uitvoerbaar. Lees elke regel — de productiekernen zijn bewust klein gehouden.


## Installatie

Voordat u deze notebook uitvoert, moet u ervoor zorgen dat u:

1. **Een Microsoft Foundry-project** heeft met een ingezet chatmodel (bijv. `gpt-4.1-mini`).
2. **Ingelogd bent met de Azure CLI** — voer `az login` uit in uw terminal.
3. **De vereiste omgevingsvariabelen hebt ingesteld:**
   - `AZURE_AI_PROJECT_ENDPOINT` — uw Microsoft Foundry-projectendpoint.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — de naam van uw ingezette model.

De RAG-sectie gebruikt **Azure AI Search** wanneer `AZURE_SEARCH_SERVICE_ENDPOINT` en `AZURE_SEARCH_API_KEY` zijn ingesteld, en valt terug op zoeken in het geheugen zodat de notebook kan draaien zonder een Search-resource.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Tools

Productietools voeren echt werk uit op echte systemen. Hier simuleren we een orderdatabase en een ticketingsysteem met eenvoudige Python-functies. De `@tool` decorator stelt ze beschikbaar voor de agent.

Merk op dat `issue_refund` `approval_mode="always_require"` gebruikt voor terugbetalingen boven een drempelwaarde — dit is de human-in-the-loop primitive die we later inzetten.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Beleid Kennisbank

Beleidsvragen ("wat is uw retourtermijn?") moeten worden beantwoord vanuit een gezaghebbende bron, niet uit het geheugen van het model. We wikkelen een kleine kennisbank in als zoekhulpmiddel.

In productie is dit **Azure AI Search**; hier bieden we een zoekfunctie op trefwoorden in het geheugen aan zodat het notitieboek overal draait, en schakelen we automatisch over naar Azure AI Search wanneer de omgevingsvariabelen aanwezig zijn.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Geheugen

Een klantenservicemedewerker die vergeet met wie hij praat, is een slechte klantenservicemedewerker. We bewaren een kleine profielwinkel per klant en voegen een korte samenvatting toe aan de instructies van de medewerker. In productie is dit een geheugendienst (zie Les 13); hier maakt een dict het patroon zichtbaar.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Model Routing en Response Caching

Twee kostenhefboomknoppen verbonden met één verzoekverwerker:

- **Routing**: een goedkope heuristische classifier beslist of een verzoek het kleine of het grote model nodig heeft.
- **Caching**: genormaliseerde herhaalde vragen worden direct vanuit een cache bediend zonder modelaanroep.

De classifier hier is opzettelijk eenvoudig. In productie zou je deze valideren tegen verkeer en zou je deze kunnen vervangen door Foundry's Model Router.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. De Agent, Menselijke Goedkeuring en Observeerbaarheid

Nu assembleren we de agent uit de bovenstaande tools en wikkelen elke aanvraag in een OpenTelemetry-span. De functie `handle_support_request` is de productieverzoekhandler: cache → route → trace → run → cache.

Menselijke goedkeuring wordt afgehandeld door het framework: omdat `issue_refund` `approval_mode="always_require"` is, pauzeert de run en toont een goedkeuringsverzoek dat we expliciet oplossen.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Evaluatiepoort

Dit is de releasepoort van de les: een offline testset beoordeelt de agent, en uitrol gaat alleen door als het slagingspercentage een drempel overschrijdt. De beoordelaar hier is een eenvoudige trefwoord-overlapcontrole om de notebook zelfvoorzienend te houden; in productie zou je een LLM-als-scheidsrechter of een framework evaluator gebruiken (zie Les 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Het Samenbrengen: Een Gesimuleerde Release

De onderstaande cel toont de hele lus die in de les wordt beschreven: voer de evaluatiepoort uit en "deploy" alleen als deze slaagt. Dit is het patroon dat je in CI zou toepassen voordat je een agentversie promoot naar de Foundry Agent Service.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Samenvatting

Je hebt een klantondersteuningsagent samengesteld die klaar is voor productie, met alle operationele zorgen erin verwerkt:

- **Tools, RAG en geheugen** geven de agent capaciteit en context.
- **Modelrouting en caching** houden latentie en kosten onder controle.
- **Menselijke goedkeuring** bewaakt risicovolle acties zoals grote terugbetalingen.
- **Het evaluatiehek** blokkeert slechte releases voordat ze worden verzonden.
- **Tracing** maakt elke aanvraag observeerbaar.

### Uitdaging

Breid deze agent uit om:

1. **Meerdere modellen te ondersteunen** — voeg een derde "redenerings"-laag toe en routeer escalaties/klachten ernaartoe.
2. **Evaluatiepoorten toe te voegen** — breid `TEST_CASES` uit om scenario's voor terugbetalingsgoedkeuring op te nemen en bevestig dat de poort regressies opvangt.
3. **Kostenbewuste routing toe te voegen** — volg een geschatte kosten per aanvraag (klein vs groot vs cache) en print een kostenrapport na een batch gemengde queries.

In de volgende les maak je de omgekeerde reis en draait een agent volledig op je eigen machine met Microsoft Foundry Local en Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
